# State Street full dataset loader

Open this notebook in JupyterLab, Jupyter Notebook or a notebook-capable editor, using a Python 3 environment with pandas installed. Extract the entire student package first and keep this notebook beside `manifest.csv` and the `data` folder. Run the cells in order.

This notebook checks delivery integrity and loads raw tables. It does not clean data, identify defects or provide a solution. No separate Python script is required.

## 1. Locate the release

If automatic discovery fails, set `ROOT` to the extracted `State_Street_Case_Student_Package` folder. The verifier uses the standard library. The loading examples require pandas.

In [ ]:
from pathlib import Path
import csv
import gzip
import hashlib

candidates = [Path.cwd(), Path.cwd() / 'State_Street_Case_Student_Package',
              Path.cwd().parent / 'State_Street_Case_Student_Package']
ROOT = next((p for p in candidates if (p / 'manifest.csv').is_file()
             and (p / 'data').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Set ROOT to the extracted State_Street_Case_Student_Package folder.')
print('Release folder:', ROOT.resolve())


## 2. Define the helpers

Raw values remain strings, including empty strings. Explicit numeric and date conversion belongs in your analysis. `iter_table` yields manageable chunks from the full table.

In [ ]:
def manifest(root=ROOT):
    with (Path(root) / 'manifest.csv').open(newline='', encoding='utf-8') as stream:
        return {row['table']: row for row in csv.DictReader(stream)}

def table_path(name, root=ROOT):
    records = manifest(root)
    if name not in records:
        raise ValueError(f'Unknown table {name!r}. Choose from {sorted(records)}')
    return Path(root) / records[name]['file']

def read_table(name, root=ROOT):
    """Return a pandas frame; preserve all values as raw strings, including blanks."""
    import pandas as pd
    return pd.read_csv(table_path(name, root), dtype=str, keep_default_na=False)

def iter_table(name, chunksize=100_000, root=ROOT):
    """Yield pandas chunks; retain a running row offset for ingestion lineage."""
    import pandas as pd
    return pd.read_csv(table_path(name, root), dtype=str,
                       keep_default_na=False, chunksize=chunksize)

def verify(root=ROOT):
    """Check transport integrity and physical counts, not data quality."""
    failures = []
    for name, expected in manifest(root).items():
        path = table_path(name, root)
        if not path.is_file():
            failures.append(f'{name}: file missing')
            continue
        sha = hashlib.sha256()
        with path.open('rb') as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b''):
                sha.update(block)
        with gzip.open(path, 'rt', newline='', encoding='utf-8') as stream:
            reader = csv.reader(stream)
            header = next(reader)
            count = sum(1 for _ in reader)
        ok = (sha.hexdigest() == expected['sha256']
              and count == int(expected['rows'])
              and len(header) == int(expected['columns'])
              and path.stat().st_size == int(expected['bytes']))
        print(f'{name}: {count:,} rows; {"PASS" if ok else "FAIL"}', flush=True)
        if not ok:
            failures.append(name)
    if failures:
        raise SystemExit('Release verification failed: ' + ', '.join(failures))
    print('All ten files verified. This does not establish correctness of their records.')



## 3. Verify all ten full tables

This reads every file and compares its checksum, row count, column count and byte size with the manifest. It can take a few minutes. A PASS establishes delivery integrity, not record correctness.

In [ ]:
verify()


## 4. Inspect the release inventory

In [ ]:
for name, row in manifest().items():
    print(f"{name}: {int(row['rows']):,} rows at {row['file']}")


## 5. Load all ten tables

This reads every table in full, each into its own DataFrame, with all values kept as raw strings. It needs roughly 1.2 GB of memory and takes about 15 to 60 seconds, depending on the machine. Each table is available by name (`transactions`, `prices` and so on) and all ten are also collected in the dictionary `data`. Row counts are checked against the manifest as they load.

In [ ]:
import pandas as pd

TABLES = ['accounts', 'securities', 'transactions', 'pipeline_events', 'positions',
          'comparison_positions', 'corporate_actions', 'prices', 'fx_rates', 'ledger_nav']

data = {}
for name in TABLES:
    data[name] = read_table(name)
    expected = int(manifest()[name]['rows'])
    assert len(data[name]) == expected, f'{name}: {len(data[name]):,} rows loaded, manifest says {expected:,}'

accounts = data['accounts']
securities = data['securities']
transactions = data['transactions']
pipeline_events = data['pipeline_events']
positions = data['positions']
comparison_positions = data['comparison_positions']
corporate_actions = data['corporate_actions']
prices = data['prices']
fx_rates = data['fx_rates']
ledger_nav = data['ledger_nav']

pd.DataFrame([{'table': name, 'rows': len(df), 'columns': df.shape[1],
               'memory_MB': round(df.memory_usage(deep=True).sum() / 1e6)}
              for name, df in data.items()]).set_index('table')

Preview the first rows of any table. This is only an on-screen view; no sample dataset is created.

In [ ]:
accounts.head()

## 6. If memory is tight, process the transaction table in chunks

If section 5 runs out of memory, restart the kernel, skip it, and work in chunks instead. This example counts every transaction row without retaining the whole table in memory. Preserve relationships and raw row lineage when designing your own pipeline.

In [ ]:
row_count = 0
for chunk in iter_table('transactions', chunksize=100_000):
    row_count += len(chunk)
print(f'Transaction rows read: {row_count:,}')
assert row_count == int(manifest()['transactions']['rows'])


## 7. Start your analysis

Read the Student Brief and Dataset Guide before joining tables. Preserve raw files and record assumptions. Develop your own checks, reconstruction, evidence traces, scorecard and dashboard. Final aggregate results must use the full release. Document dependencies, runtime and AI use. The Working Guide has optional advice on approach; it contains no results for this dataset.